In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from joblib import Parallel, delayed

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "evaluation").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from evaluation.split import leave_last_out_split
from evaluation.metrics import recall_at_k, ndcg_at_k

from models.baselines import PopularityRecommender
from models.content_based import ContentBasedRecommender, HybridMFContentRecommender
from models.mf import BPRMatrixFactorization

print("=== 1. PREPARING DATA")
interactions = pd.read_csv("../data/raw/RAW_interactions.csv")
recipes = pd.read_csv("../data/raw/RAW_recipes.csv")

interactions["date"] = pd.to_datetime(interactions["date"], errors="coerce")
positive_df = interactions[interactions["rating"] >= 4].copy().rename(columns={"date": "date_time"})
user_counts = positive_df["user_id"].value_counts()
positive_df = positive_df[positive_df["user_id"].isin(user_counts[user_counts >= 2].index)].copy()

recipes_model = recipes.rename(columns={"id": "recipe_id"}).copy()
df_model = positive_df.merge(recipes_model[["recipe_id"]], on="recipe_id", how="inner")

train_df, test_df = leave_last_out_split(df_model, user_col="user_id", time_col="date_time")
train_history = train_df.groupby("user_id")["recipe_id"].apply(set).to_dict()

# Finde die Cold Items (Items im Test-Set, die im Training fehlen)
train_items_set = set(train_df["recipe_id"].unique())
test_items_set = set(test_df["recipe_id"].unique())
cold_items = test_items_set - train_items_set

true_item_by_user = test_df.set_index("user_id")["recipe_id"].to_dict()

# Wir evaluieren NUR User, deren Test-Item "cold" ist
cold_test_users = [u for u, i in true_item_by_user.items() if i in cold_items]
print(f"Total Cold Items in Test Set: {len(cold_items)}")
print(f"Users to evaluate (Cold-Item target): {len(cold_test_users)}")


print("\n=== 2. TRAINING ===")
pop_model = PopularityRecommender()
pop_model.fit(train_df, item_col="recipe_id")

bpr_model = BPRMatrixFactorization(k_factors=32, epochs=5)
bpr_model.fit(train_df, item_col="recipe_id")

content_model = ContentBasedRecommender(
    text_cols=["name", "tags", "ingredients", "description"],
    metadata_cols=["minutes", "n_steps", "n_ingredients"],
    min_df=2,
    ngram_range=(1, 1),
    time_col="date_time",
    recency_decay=0.0,
)
content_model.fit(train_df, recipes_model, user_col="user_id", item_col="recipe_id")

hybrid_model = HybridMFContentRecommender(
    mf_model=bpr_model,
    content_model=content_model,
    alpha=0.9,
    adaptive=True,
    sparse_threshold=5
)

models = {
    "Popularity": pop_model,
    "BPR MF": bpr_model,
    "Content-Based": content_model,
    "Hybrid BPR+Content": hybrid_model
}

print("\n=== 3. EVALUATING TRUE COLD-START RECALL ===")
K = 20

def evaluate_cold_user(user):
    true_item = true_item_by_user[user]
    user_hist = train_history.get(user, set())

    user_results = {}
    for name, model in models.items():
        recs = model.recommend(user, user_history=user_hist, k=K)
        user_results[name] = {
            "recall": recall_at_k(recs, [true_item], k=K),
            "ndcg": ndcg_at_k(recs, [true_item], k=K)
        }
    return user_results

parallel_results = Parallel(n_jobs=-1, batch_size=50)(
    delayed(evaluate_cold_user)(user) for user in cold_test_users
)

final_metrics = {name: {"recalls": [], "ndcgs": []} for name in models.keys()}

for user_res in parallel_results:
    for name in models.keys():
        final_metrics[name]["recalls"].append(user_res[name]["recall"])
        final_metrics[name]["ndcgs"].append(user_res[name]["ndcg"])

print("\n=== FINAL COLD-ITEM SLICE RESULTS ===")
print(f"{'Model':<20} | {'Recall@20':<10} | {'NDCG@20':<10}")
print("-" * 45)

for name in models.keys():
    mean_rec = np.mean(final_metrics[name]["recalls"])
    mean_ndcg = np.mean(final_metrics[name]["ndcgs"])
    print(f"{name:<20} | {mean_rec:.4f}     | {mean_ndcg:.4f}")

=== 1. PREPARING DATA
Sorting data chronologically to prevent time leakage...
Dropping users with tied final timestamps because a strict leave-last-out split is not identifiable: 8115 users
Splitting data: Leave-last-out...
Train Set: 707232 interactions
Test Set: 44249 interactions
Total Cold Items in Test Set: 4633
Users to evaluate (Cold-Item target): 4869

=== 2. TRAINING ===
Training Popularity Baseline...
Learned popularity ranking for 185226 unique recipes.
Training BPR MF (k=32, epochs=5)...
Epoch 1/5 completed.
Epoch 2/5 completed.
Epoch 3/5 completed.
Epoch 4/5 completed.
Epoch 5/5 completed.
BPR Training complete!
Training Content-Based Recommender (with proper Cold-Start handling)...
Using text columns: ['name', 'tags', 'ingredients', 'description']
Using metadata columns: ['minutes', 'n_steps', 'n_ingredients']
Feature dimension: 45344
Vocabulary size: 45344
Missing rates by feature: {'name': 4.317099599804867e-06, 'tags': 0.0, 'ingredients': 0.0, 'description': 0.02149483